# CAGED Indústria SP — Download e Pré-processamento

**Fonte:** Novo CAGED / Ministério do Trabalho e Emprego
**Filtro:** UF = 35 (São Paulo) · Seção C (Indústria de Transformação)
**Período:** Jan/2021 – Dez/2023

> ⚠️ Execute este notebook antes do `02_analise_sql.ipynb`.
> O download completo (~2GB) pode levar 30-60 minutos.

In [ ]:
import os
import requests
import zipfile
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

ANOS = [2021, 2022, 2023]
UF_FILTRO = 35
SECAO_FILTRO = "C"


In [ ]:
MUNICIPIOS_SP = {
    "355030": "São Paulo", "352900": "Guarulhos", "354340": "Osasco",
    "354870": "Santo André", "355220": "Sorocaba", "351880": "Campinas",
    "354850": "São Bernardo do Campo", "354980": "São José dos Campos",
    "353440": "Ribeirão Preto", "354910": "São José do Rio Preto",
    "355100": "Santos", "351570": "Bauru", "350950": "Caçapava",
    "354515": "Santo André", "350280": "Americana", "352310": "Itaquaquecetuba",
    "350800": "Cajamar", "352220": "Indaiatuba", "354050": "Piracicaba",
    "354130": "Presidente Prudente"
}

GRAU_INSTRUCAO = {
    1: "Analfabeto", 2: "Até 5ª incompleto", 3: "5ª completo fund.",
    4: "6ª a 9ª fund.", 5: "Fund. completo", 6: "Médio incompleto",
    7: "Médio completo", 8: "Superior incompleto", 9: "Superior completo",
    10: "Mestrado", 11: "Doutorado"
}


In [ ]:
def download_caged_mes(ano: int, mes: int, dest_dir: Path) -> Path | None:
    """Baixa e extrai arquivo mensal do Novo CAGED do FTP do MTE."""
    nome_arquivo = f"CAGEDMOV{ano}{mes:02d}.txt"
    dest_path = dest_dir / nome_arquivo

    if dest_path.exists():
        print(f"  Já existe: {nome_arquivo}")
        return dest_path

    url = f"ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/{ano}/{ano}{mes:02d}/CAGEDMOV{ano}{mes:02d}.7z"

    print(f"  Baixando: {url}")
    try:
        response = requests.get(url, timeout=120, stream=True)
        response.raise_for_status()

        temp_path = dest_dir / f"temp_{ano}{mes:02d}.7z"
        with open(temp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        import py7zr
        with py7zr.SevenZipFile(temp_path, mode="r") as z:
            z.extractall(dest_dir)
        temp_path.unlink()

        return dest_path
    except Exception as e:
        print(f"  ERRO: {e}")
        return None


In [ ]:
# ⚠️ EXECUTAR MANUALMENTE — Download pode levar 30-60 minutos
# Descomente e rode quando pronto para baixar os dados reais

# arquivos_baixados = []
# for ano in ANOS:
#     for mes in range(1, 13):
#         path = download_caged_mes(ano, mes, DATA_DIR)
#         if path:
#             arquivos_baixados.append(path)
# print(f"Total de arquivos baixados: {len(arquivos_baixados)}")
print("Célula de download — descomente para executar")


In [ ]:
def filtrar_e_consolidar(data_dir: Path, uf: int, secao: str) -> pd.DataFrame:
    arquivos = list(data_dir.glob("CAGEDMOV*.txt"))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo CAGED encontrado em {data_dir}. Execute a célula de download primeiro.")

    dfs = []
    for arq in sorted(arquivos):
        df = pd.read_csv(
            arq, sep=";", encoding="latin-1", low_memory=False,
            usecols=["competência", "uf", "município", "seção", "subseção",
                     "salárioMensal", "sexo", "grauDeInstrução", "raçaCor",
                     "saldoMovimentação"]
        )
        df.columns = ["competencia", "uf", "municipio_cod", "secao", "subsecao",
                      "salario_mensal", "sexo", "grau_instrucao", "raca_cor", "saldo_mov"]

        df = df[(df["uf"] == uf) & (df["secao"] == secao)].copy()
        df["competencia"] = pd.to_datetime(df["competencia"].astype(str), format="%Y%m")
        df["municipio_cod"] = df["municipio_cod"].astype(str).str.zfill(6)

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


# df_caged = filtrar_e_consolidar(DATA_DIR, UF_FILTRO, SECAO_FILTRO)
# df_caged.to_parquet(DATA_DIR / "caged_sp_industria.parquet", index=False)
# print(f"Salvo: {len(df_caged):,} registros")
print("Célula de consolidação — descomente após download concluído")


In [ ]:
schema_exemplo = pd.DataFrame({
    "competencia":    ["2021-01-01", "2021-01-01", "2021-02-01"],
    "municipio_cod":  ["355030",     "352900",     "355030"],
    "secao":          ["C",          "C",          "C"],
    "saldo_mov":      [1240,         -85,          970],
    "salario_mensal": [3200.50,      2890.00,      3100.75],
    "sexo":           [1,            2,            1],
    "grau_instrucao": [7,            7,            8],
})
print("Schema esperado após processamento:")
print(schema_exemplo.dtypes)
print()
schema_exemplo
